# TP6 — Optimiser (2/2) : hyperparamètres et compression
**ÉNONCÉ** · Durée estimée : **une journée**

## 🎯 Ce que vous allez apprendre
- Distinguer **paramètres** (appris) et **hyperparamètres** (choisis par vous)
- Faire une **recherche aléatoire** d'hyperparamètres, et la **valider** sur plusieurs graines
- **Compresser** un modèle : élagage (pruning) et quantification (TensorFlow Lite)
- Juger un **compromis précision / taille**, avec des chiffres

## 🧰 Boîte à outils
Noms de fonctions et de classes utiles. **Aucun exemple d'appel n'est donné** : cherchez dans la documentation (scikit-learn, Keras, NumPy, matplotlib) comment les utiliser.

`random.choice`  ·  `random.seed`  ·  `pd.DataFrame.sort_values`  ·  `keras_tuner.RandomSearch`  ·  `np.quantile`  ·  `clone_model`  ·  `set_weights`  ·  `tf.lite.TFLiteConverter`  ·  `tf.lite.Interpreter`  ·  `os.path.getsize`

## 📘 Comment travailler
- Les cellules **déjà écrites** sont à lire et à exécuter.
- Les cellules **🎯 Étape** sont à **écrire vous-même**, à partir de la consigne.
- Après chaque étape, une cellule **✅ Contrôle** vérifie votre résultat. Respectez les **noms de variables** demandés.
- Les questions **✍️** demandent une réponse écrite : c'est là que vous montrez que vous avez compris.
- Bloqué ? Relisez l'indice, puis cherchez dans la documentation. Un message d'erreur se lit : la dernière ligne dit souvent ce qui ne va pas.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, confusion_matrix, precision_score, recall_score
from tensorflow import keras
from tensorflow.keras import layers, regularizers
keras.utils.set_random_seed(42)
import random, tensorflow as tf

def controle(condition, ok, ko):
    print("✅", ok) if condition else print("❌", ko)

## Étape 1 — Préparer : un jeu de validation et une fonction d'essai
Pour choisir des hyperparamètres, on compare **sur un jeu de validation, jamais sur le test** : sinon le test n'est plus « jamais vu » et son verdict est faussé. On découpe donc l'apprentissage en deux. La fonction `essayer` entraîne un réseau avec les réglages donnés et renvoie sa **précision de validation**.

In [2]:
from sklearn.datasets import load_digits
chiffres = load_digits()
X = chiffres.data / 16.0
y = chiffres.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)

def essayer(neurones=32, couches=2, dropout=0.0, lr=1e-3, batch=32, seed=42):
    keras.utils.set_random_seed(seed)
    m = keras.Sequential([layers.Input(shape=(64,))])
    for _ in range(couches):
        m.add(layers.Dense(neurones, activation="relu"))
        if dropout:
            m.add(layers.Dropout(dropout))
    m.add(layers.Dense(10, activation="softmax"))
    m.compile(optimizer=keras.optimizers.Adam(lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    arret = keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    m.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=40, batch_size=batch, callbacks=[arret], verbose=0)
    acc_val = float(np.mean(np.argmax(m.predict(X_val, verbose=0), axis=1) == y_val))
    return acc_val, m

acc_val_defaut, _ = essayer()
bruit_val = 1.96 * np.sqrt(acc_val_defaut * (1 - acc_val_defaut) / len(y_val))
print(f"Réglage par défaut : précision de validation = {acc_val_defaut:.3f}")
print(f"Avec {len(y_val)} images de validation, une différence inférieure à environ {bruit_val:.3f} est dans le bruit.")

Réglage par défaut : précision de validation = 0.963
Avec 270 images de validation, une différence inférieure à environ 0.023 est dans le bruit.


## 🎯 Étape 2 — Explorer le taux d'apprentissage
Le **taux d'apprentissage** est la taille du pas de chaque correction. C'est souvent **l'hyperparamètre le plus important**.

**À faire :** Avec `essayer`, testez les taux **0,0001 / 0,001 / 0,01 / 0,1** (autres réglages par défaut). Rangez les précisions de validation dans un dictionnaire `scores_lr` (clé = taux) et tracez-les (axe horizontal logarithmique).

> 💡 **Indice** : `plt.xscale("log")`. Une boucle `for lr in (...)`.

In [3]:
scores_lr = {}
# DONE
for lr in (0.0001, 0.001, 0.01, 0.1):
    acc_val, _ = essayer(lr=lr)
    scores_lr[lr] = acc_val
print(scores_lr)

{0.0001: 0.8629629629629629, 0.001: 0.9629629629629629, 0.01: 0.9740740740740741, 0.1: 0.8629629629629629}


In [4]:
# ✅ Contrôle (ne pas modifier)
controle(len(scores_lr) == 4, '4 taux testés', 'Attendu 4 résultats')

✅ 4 taux testés


### ✍️ Question
**Que se passe-t-il aux deux extrêmes (taux très petit, taux très grand) ? Les écarts dépassent-ils le bruit mesuré plus haut ?**

*Votre réponse (2 à 4 lignes) :*

> La précision se dégrade aux deux extrémités, avec un écart sensiblement supérieur au bruit par rapport à la précision aux valeurs centrales qui elles sont proches (écart inférieur au bruit)

## 🎯 Étape 3 — Recherche aléatoire
Tester toutes les combinaisons (une **grille**) explose vite : 4 × 3 × 5 × 3 × 3 = 540 réseaux. La **recherche aléatoire** en tire seulement quelques-unes au hasard, et marche souvent aussi bien à budget égal.

**À faire :** **D'abord**, tirez **15 configurations au hasard** (après `random.seed(0)`) dans une liste `configs`, parmi : `neurones` ∈ {16, 32, 64, 128}, `couches` ∈ {1, 2, 3}, `dropout` ∈ {0, 0.1, 0.2, 0.3, 0.5}, `lr` ∈ {0.001, 0.003, 0.01}, `batch` ∈ {16, 32, 64}. **Ensuite**, évaluez chacune avec `essayer`. Rangez le tout dans un tableau **`recherche`** (une colonne par réglage + `acc_val`), **trié par `acc_val` décroissant**. Vérifiez que vos 15 configurations sont bien **différentes**.

> 💡 **Indice** : `random.choice(liste)`. `pd.DataFrame(liste_de_dicts).sort_values(..., ascending=False)`. ⚠️ **Piège** : `keras.utils.set_random_seed` (appelé dans `essayer`) réinitialise **aussi** le module `random` de Python. Si vous tirez une configuration, puis entraînez, puis retirez, vous retomberez toujours sur la même. D'où l'ordre : tirer tout **avant** d'entraîner.

In [ ]:
random.seed(0)
# DONE : d'abord tirer les 15 configurations (liste configs), ensuite les évaluer
configs = []
while len(configs) < 15:
    neurones = random.choice([16, 32, 64, 128])
    couches = random.choice([1, 2, 3])
    dropout = random.choice([0, 0.1, 0.2, 0.3, 0.5])
    lr = random.choice([0.001, 0.003, 0.01])
    batch = random.choice([16, 32, 64])
    nouveau = {"neurones":neurones, "couches":couches, "dropout":dropout, "lr":lr, "batch":batch, "acc_val":0}
    if nouveau not in configs:
        configs.append(nouveau)
for conf in configs:
    conf["acc_val"], _ = essayer(neurones=conf["neurones"], couches=conf["couches"], dropout=conf["dropout"], lr=conf["lr"], batch=conf["batch"])
recherche = pd.DataFrame(configs)
recherche.sort_values(by="acc_val", ascending=False, inplace=True)
recherche.head(15)

,neurones,couches,dropout,lr,batch,acc_val
2,32,3,0.1,0.003,16,0.981481
7,128,2,0.5,0.003,16,0.981481
0,128,2,0.0,0.003,64,0.977778
12,16,1,0.2,0.010,32,0.977778
1,128,2,0.2,0.003,32,0.977778
10,64,3,0.0,0.001,64,0.974074
4,32,2,0.0,0.010,16,0.966667
5,64,2,0.5,0.001,32,0.966667
6,128,2,0.5,0.010,16,0.962963
11,32,1,0.1,0.010,32,0.962963


In [16]:
# ✅ Contrôle (ne pas modifier)
nb_differentes = len(recherche.drop(columns='acc_val').drop_duplicates())
controle(len(recherche) == 15 and recherche['acc_val'].is_monotonic_decreasing and nb_differentes >= 12, '15 essais triés, %d configurations différentes' % nb_differentes, 'Attendu 15 lignes triées et au moins 12 configurations différentes (avez-vous tiré avant d\'entraîner ?)')

✅ 15 essais triés, 15 configurations différentes


### ✍️ Question
**Quel écart y a-t-il entre le meilleur et le pire essai ? Le meilleur est-il réellement meilleur que le réglage par défaut, compte tenu du bruit ?**

*Votre réponse (2 à 4 lignes) :*

> L'écart meilleur - pire et d'environ 15%. Le meilleur résultat est 2% supérieur au réglage par défaut, ce qui n'est pas supérieur au bruit, il est donc équivalent au réglage par défaut.

## Étape 4 — Bonus : la même recherche avec Keras Tuner
Un outil dédié automatise la recherche et garde la trace des essais. **À installer** : `pip install keras-tuner tensorboard`. Cette cellule est un bonus : si elle ne s'exécute pas, la suite du TP n'en dépend pas.

In [17]:
try:
    import keras_tuner as kt
    def construire_hp(hp):
        m = keras.Sequential([layers.Input(shape=(64,))])
        for _ in range(hp.Int("couches", 1, 3)):
            m.add(layers.Dense(hp.Choice("neurones", [16, 32, 64, 128]), activation="relu"))
            m.add(layers.Dropout(hp.Choice("dropout", [0.0, 0.1, 0.2, 0.3, 0.5])))
        m.add(layers.Dense(10, activation="softmax"))
        m.compile(optimizer=keras.optimizers.Adam(hp.Choice("lr", [1e-3, 3e-3, 1e-2])), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        return m
    tuner = kt.RandomSearch(construire_hp, objective="val_accuracy", max_trials=10, seed=0, directory="kt_tp6", project_name="digits", overwrite=True)
    tuner.search(X_tr, y_tr, validation_data=(X_val, y_val), epochs=30, batch_size=32,
                 callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)], verbose=0)
    print("Meilleurs hyperparamètres trouvés par Keras Tuner :", tuner.get_best_hyperparameters(1)[0].values)
except ImportError:
    print("keras-tuner n'est pas installé : bonus ignoré (pip install keras-tuner tensorboard)")


Meilleurs hyperparamètres trouvés par Keras Tuner : {'couches': 3, 'neurones': 128, 'dropout': 0.2, 'lr': 0.003}


## 🎯 Étape 5 — Valider sur plusieurs graines
Le meilleur essai de la recherche a pu avoir de la **chance**. Pour le savoir, on **répète** : la meilleure configuration et la configuration par défaut, **5 fois chacune** avec des graines différentes.

**À faire :** Récupérez la meilleure configuration de `recherche` (1re ligne) dans `meilleure` (dictionnaire des 5 réglages, sans `acc_val`). Avec `essayer`, calculez les précisions de validation de la configuration par défaut (`scores_defaut`) puis de la meilleure (`scores_meilleure`) pour les graines **0 à 4**. Calculez `gain` (différence des moyennes) et affichez une **conclusion automatique** : « réel » si `gain > bruit_val`, sinon « dans le bruit ».

> 💡 **Indice** : `recherche.iloc[0]`, `.to_dict()`. Attention aux types : `int(...)` pour `neurones`, `couches`, `batch`.

In [ ]:
# DONE : meilleure, scores_defaut, scores_meilleure, gain, conclusion
meilleure = recherche.drop(columns='acc_val').iloc[0]
scores_defaut = []
scores_meilleure = []
for seed in range(5):
    acc, _ = essayer(seed=seed)
    scores_defaut.append(acc)
    acc, _ = essayer(neurones=int(meilleure["neurones"]), couches=int(meilleure["couches"]), dropout=meilleure["dropout"], lr=meilleure["lr"], batch=int(meilleure["batch"]), seed=seed)
    scores_meilleure.append(acc)
gain = np.mean(scores_meilleure) - np.mean(scores_defaut)
if gain > bruit_val:
    conclusion = "réel"
else:
    conclusion = "dans le bruit"
print(gain, conclusion)

0.007407407407407418 dans le bruit


In [27]:
# ✅ Contrôle (ne pas modifier)
controle(len(scores_defaut) == 5 and len(scores_meilleure) == 5 and 'neurones' in meilleure, 'Deux séries de 5 graines', 'Attendu 5 valeurs dans chaque liste et un dictionnaire meilleure')

✅ Deux séries de 5 graines


### ✍️ Question
**L'optimisation d'hyperparamètres a-t-elle apporté un gain démontré ? Est-ce un échec si ce n'est pas le cas ?**

*Votre réponse (2 à 4 lignes) :*

> La configuration par défaut a donné des bons résultats, de sorte que l'optimisation n'a guère pu les améliorer. La démarche reste valide dans la mesure où la configuration par défaut ne sera pas toujours bonne d'une part, et d'autre part la connaissance de l'optimum est un avantage pour valider l'hyperparamétrage choisi.

## 🎯 Étape 6 — Évaluer le modèle final, une seule fois sur le test
On entraîne la meilleure configuration retenue, puis on mesure **une seule fois** sur le jeu de test, resté intact jusqu'ici.

**À faire :** Entraînez la meilleure configuration avec `essayer(**meilleure)` (graine 42) pour obtenir `modele_final`. Calculez `acc_test_final` sur `X_test`.

> 💡 **Indice** : `essayer` renvoie un couple `(précision de validation, modèle)`.

In [ ]:
# DONE : modele_final et acc_test_final
acc, modele_final = essayer(neurones=int(meilleure["neurones"]), couches=int(meilleure["couches"]), dropout=meilleure["dropout"], lr=meilleure["lr"], batch=int(meilleure["batch"]))

In [32]:
acc_test_final = np.mean(np.argmax(modele_final.predict(X_test, verbose=0), axis=1) == y_test)

In [33]:
# ✅ Contrôle (ne pas modifier)
controle(acc_test_final > 0.9, 'Modèle final : %.3f sur le test' % acc_test_final, 'Attendu plus de 0.9')

✅ Modèle final : 0.978 sur le test


## 🎯 Étape 7 — Compression 1 : l'élagage
L'**élagage** (pruning) met à zéro les poids les plus proches de zéro : ils contribuent peu. Un modèle avec beaucoup de zéros peut être stocké et calculé plus efficacement. Jusqu'où peut-on aller sans perdre en précision ?

**À faire :** Pour des taux **0, 50, 80, 90 et 95 %**, mettez à zéro les poids (matrices seulement, pas les biais) dont la **valeur absolue est sous le quantile** correspondant, puis mesurez la précision test du modèle élagué. Rangez dans un dictionnaire `resultats_elagage` (clé = taux, de 0 à 0.95).

> 💡 **Indice** : `modele_final.get_weights()` renvoie une liste ; les matrices ont `ndim == 2`. `np.quantile(np.abs(...), taux)`. Pour évaluer un jeu de poids : `keras.models.clone_model(modele_final)` puis `set_weights`.

In [ ]:
# DONE : resultats_elagage
resultats_elagage = {}
resultats_elagage[0] = acc_test_final
for taux in (0.5, 0.8, 0.9, 0.95):
    weights = modele_final.get_weights()
    res = [np.abs(x) for x in weights[::2]] # les lignes paires correspondent aux poids, les impaires aux biais
    res = np.concatenate([x.ravel() for x in res]) # on met tous les poids (en valeur absolue) dans un seul vecteur
    seuil = np.quantile(res,taux)
    for couche in range(4):
        for neurone in weights[2*couche]:
            for i in range(len(neurone)):
                if np.abs(neurone[i]) < seuil:
                    neurone[i] = 0
    modele_allege = keras.models.clone_model(modele_final)
    modele_allege.set_weights(weights)
    resultats_elagage[taux] = np.mean(np.argmax(modele_allege.predict(X_test, verbose=0), axis=1) == y_test)

In [94]:
print(resultats_elagage)

{0: np.float64(0.9777777777777777), 0.5: np.float64(0.9533333333333334), 0.8: np.float64(0.5644444444444444), 0.9: np.float64(0.10888888888888888), 0.95: np.float64(0.1)}


0.5 est un taux acceptable (écart à la limite du bruit), un taux supérieur ne conviendrait pas.

In [95]:
# ✅ Contrôle (ne pas modifier)
controle(abs(resultats_elagage[0] - acc_test_final) < 1e-9 and resultats_elagage[0.95] <= resultats_elagage[0], 'Élagage cohérent : 0 % = modèle original', 'Attendu : 0 % identique au modèle final, et 95 % pas meilleur')       

✅ Élagage cohérent : 0 % = modèle original


## 🎯 Étape 8 — Compression 2 : la quantification
La **quantification** stocke les poids sur **8 bits entiers** au lieu de 32 bits décimaux : le fichier est environ **4 fois plus petit** et le calcul plus rapide. C'est la technique de compression la plus utilisée. TensorFlow Lite la fait en quelques lignes.

**À faire :** Convertissez `modele_final` en modèle TensorFlow Lite quantifié (`tf.lite.Optimize.DEFAULT`). Mesurez `taille_keras` (taille en octets du modèle sauvegardé au format `.keras`) et `taille_tflite`. Vérifiez la précision du modèle quantifié sur le test avec l'interpréteur TFLite : `acc_tflite`.

> 💡 **Indice** : `modele_final.save("fichier.keras")`, `os.path.getsize`. `tf.lite.TFLiteConverter.from_keras_model`, `.optimizations`, `.convert()`. `tf.lite.Interpreter(model_content=...)`, `allocate_tensors`, `set_tensor`, `invoke`, `get_tensor`.

In [ ]:
# DONE : taille_keras, taille_tflite, acc_tflite
modele_final.save("modele_final.keras")
taille_keras = os.path.getsize("modele_final.keras")

In [99]:
from sklearn.metrics import accuracy_score

converter = tf.lite.TFLiteConverter.from_keras_model(modele_final)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
modele_tflite = converter.convert()

interpreter = tf.lite.Interpreter(model_content=modele_tflite)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

y_pred_tflite = []
for x in X_test:
    # Ajout de la dimension batch
    x = np.expand_dims(x, axis=0).astype(input_details[0]["dtype"])

    interpreter.set_tensor(input_details[0]["index"], x)
    interpreter.invoke()

    prediction = interpreter.get_tensor(output_details[0]["index"])
    y_pred_tflite.append(np.argmax(prediction[0]))

acc_tflite = accuracy_score(y_test, y_pred_tflite)



INFO:tensorflow:Assets written to: C:\Users\UTILIS~1\AppData\Local\Temp\tmplj9vows0\assets


INFO:tensorflow:Assets written to: C:\Users\UTILIS~1\AppData\Local\Temp\tmplj9vows0\assets


Saved artifact at 'C:\Users\UTILIS~1\AppData\Local\Temp\tmplj9vows0'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 64), dtype=tf.float32, name='keras_tensor_218')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  2895629990512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2895650188384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2895654088496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2895654079136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2895629979952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2895650185504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2895654087056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2895654078896: TensorSpec(shape=(), dtype=tf.resource, name=None)


c:\Users\Utilisateur\AppData\Local\Programs\Python\Python314\Lib\site-packages\tensorflow\lite\python\interpreter.py:468: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [103]:
with open("modele_quantifie.tflite", "wb") as f:
    f.write(modele_tflite)
taille_tflite = os.path.getsize("modele_quantifie.tflite")

In [104]:
# ✅ Contrôle (ne pas modifier)
controle(taille_tflite < taille_keras and abs(acc_tflite - acc_test_final) < 0.03, 'Modèle plus petit, précision conservée', 'Attendu : un fichier plus petit et une précision proche (écart < 0.03)')

✅ Modèle plus petit, précision conservée


### ✍️ Question
**Quel compromis choisiriez-vous entre taille, vitesse et précision pour mettre ce modèle en production ? Justifiez avec vos chiffres.**

*Votre réponse (2 à 4 lignes) :*

> A précision équivalente, le mieux est de choisir un modèle avec le moins de paramètres possible, élagué et quantifié. Plus un modèle est léger, plus il est rapide. Bien entendu la précision ainsi obtenue doit être proche de celle de l'optimum (près de 98%, avec un bruit autour de 2.5%)

## ✅ Ce qu'il faut retenir
- **Paramètres** = appris par le modèle. **Hyperparamètres** = choisis par vous (taux d'apprentissage, neurones, couches, dropout, batch…).
- On les compare **sur la validation, jamais sur le test**, et on **répète sur plusieurs graines**.
- Un gain ne compte que s'il dépasse le **bruit**. Ne pas trouver de gain est un résultat, pas un échec.
- **Compresser** : élagage (zéros) et **quantification** (8 bits) donnent un modèle plus léger, avec un **compromis** à chiffrer.
- Le jeu de test sert **une seule fois, à la fin**.

## 📋 Tableau de synthèse à compléter (dans votre compte rendu)
| Version du modèle | Précision test | Taille | Commentaire |
|---|---|---|---|
| Modèle de départ (défaut) | 0.96 | | |
| Meilleure configuration | 0.98 | 90 Ko | bruit |
| Élagué à 50 % | 0.95 | | limite du bruit |
| Quantifié (TFLite) | 0.98 | 10 Ko | bruit |